# Final Production Colab Run
Notebook ini untuk run final IndoBERT+BiLSTM (config terbaik) secara langsung di Colab T4.

## 0) Setup Runtime
Pilih GPU T4 di Colab: Runtime > Change runtime type > T4 GPU.

In [ ]:
!git clone https://github.com/kzquandary/mbg-sentiment.git
%cd mbg-sentiment
!git checkout main
!python3 -m pip install -r requirements.txt
!nvidia-smi

## 1) Validasi Dataset & Config
Dataset dan config final sudah ada di repo.

In [ ]:
import os
assert os.path.exists('data/dataset_relabel_mbg_improved.csv'), 'Dataset improved tidak ditemukan'
assert os.path.exists('src/resources/step7_final_production.json'), 'Config final tidak ditemukan'
print('OK dataset:', 'data/dataset_relabel_mbg_improved.csv')
print('OK config :', 'src/resources/step7_final_production.json')

## 2) Persiapan Data (CPU)
- Split 70/30
- Preprocess train/test
- Split internal train_sub/val_sub
- Balancing train_sub (fixed 1500 per kelas)

In [ ]:
!python3 src/05_split_data.py --input data/dataset_relabel_mbg_improved.csv --text-col text --label-col Labeling_Sentimen --train-ratio 0.7 --val-ratio 0 --test-ratio 0.3
!python3 src/03_preprocess_text.py --train-input data/train.csv --test-input data/test.csv --text-col text

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('data/train.csv')
train_sub, val_sub = train_test_split(df, test_size=0.15, random_state=42, stratify=df['Labeling_Sentimen'].astype(str))
train_sub.to_csv('data/train_sub.csv', index=False, encoding='utf-8-sig')
val_sub.to_csv('data/val_sub.csv', index=False, encoding='utf-8-sig')
print('train_sub', len(train_sub), train_sub['Labeling_Sentimen'].value_counts().to_dict())
print('val_sub  ', len(val_sub), val_sub['Labeling_Sentimen'].value_counts().to_dict())

In [ ]:
!python3 src/05b_balance_train.py --input data/train_sub.csv --label-col Labeling_Sentimen --target-by-label-json src/resources/train_balance_push80_valid.json --output data/train_sub_balanced.csv --log-output outputs/train_sub_balance_log.json

## 3) Train Final Model (GPU)

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:64'
!python3 src/07_indobert_bilstm.py --train data/train_sub_balanced.csv --val data/val_sub.csv --trial-configs-json src/resources/step7_push80_valid_trials.json --max-trials 3

## 4) Tune Class Multiplier (Val) + Evaluate Test
Langkah ini melakukan threshold-like tuning berbasis probabilitas untuk menaikkan macro-F1 kelas minoritas.

In [ ]:
!python3 src/09b_tune_class_multipliers.py --val data/val_sub.csv --model-path models/best_indobert_bilstm.pt --output outputs/best_class_multipliers.json --summary-output outputs/class_multiplier_tuning_summary.json
!python3 src/09_evaluate.py --test data/test.csv --model-path models/best_indobert_bilstm.pt --class-multiplier-json outputs/best_class_multipliers.json

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image

metrics = json.load(open('outputs/final_metrics.json', encoding='utf-8'))
report = pd.read_csv('outputs/classification_report.csv')
cm = pd.read_csv('outputs/confusion_matrix.csv', index_col=0)

print('Final Metrics')
display(pd.DataFrame([metrics])[['accuracy','precision_macro','recall_macro','f1_macro']])
print('Classification Report')
display(report)
print('Confusion Matrix (table)')
display(cm)
print('Confusion Matrix (figure)')
display(Image(filename='outputs/figures/confusion_matrix.png'))

## 5) Output Final
Artifact utama:
- `models/best_indobert_bilstm.pt`
- `outputs/train_sub_balance_log.json`
- `outputs/step7_trials.csv`
- `outputs/step7_best_config.json`
- `outputs/best_class_multipliers.json`
- `outputs/class_multiplier_tuning_summary.json`
- `outputs/final_metrics.json`
- `outputs/classification_report.csv`
- `outputs/confusion_matrix.csv`
- `outputs/figures/confusion_matrix.png`